# 🤖 AI Engineering Fundamentals — Lezione 3
## Notebook Gruppo A

**ITS Novitas 4.0 | Martedì 26/05/2026**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [ ]:
GRUPPO = "A"
MEMBRI = ["", "", "", ""]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

In [ ]:
!pip install anthropic -q
from google.colab import userdata
import anthropic, os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

def chiedi_claude(messaggio, temperature=0.7, system=None, max_tokens=500):
    params = {
        "model": "claude-haiku-4-5-20251001",
        "max_tokens": max_tokens,
        "temperature": temperature,
        "messages": [{"role": "user", "content": messaggio}]
    }
    if system:
        params["system"] = system
    return client.messages.create(**params)

print("✅ Setup completato!")

---
## 🎯 Tema del Gruppo A: Context Window & Token

Esplorate il limite fondamentale di ogni LLM: la context window.
Misurate quanto 'pesa' una conversazione e quanto costa.

---
### Esercizio 1 — Misurare la context window in tempo reale *(guidato)*

Costruite una conversazione di 5 turni e misurate
quanti token occupa la history dopo ogni messaggio.

In [ ]:
# Esercizio 1 — misurare i token della history

history = []

def conta_token_history(history):
    """Conta i token della history attuale senza inviare al modello."""
    if not history:
        return 0
    result = client.messages.count_tokens(
        model="claude-haiku-4-5-20251001",
        messages=history
    )
    return result.input_tokens

domande = [
    "Ciao! Sono uno studente di AI Engineering a Sassari.",
    "Cosa sono i sensori IoT?",
    "Come si connettono al cloud?",
    "Quali dati raccolgono tipicamente?",
    "Come vengono analizzati i dati?",
]

print(f"{'Turno':<8} {'Messaggi':<12} {'Token history':<18} {'Costo stimato ($)':<20}")
print("-" * 60)

for i, domanda in enumerate(domande):
    history.append({"role": "user", "content": domanda})

    # TODO: chiamate chiedi_claude con messages=history
    risposta = ___
    history.append({"role": "assistant", "content": risposta.content[0].text})

    token = conta_token_history(history)
    # TODO: calcolate il costo stimato (input: $1/M token)
    costo = ___

    print(f"{i+1:<8} {len(history):<12} {token:<18} ${costo:.6f}")

print()
print("💡 I token crescono linearmente con la conversazione?")
# Osservazione: ...

---
### Esercizio 2 — Italiano vs Inglese: quanto pesa la lingua? *(guidato)*

La stessa conversazione in italiano vs inglese.
Quanti token in più occupa la versione italiana?

In [ ]:
# Esercizio 2 — confronto token italiano vs inglese

conversazione_it = [
    {"role": "user", "content": "Cos'è il monitoraggio ambientale con sensori IoT?"},
    {"role": "assistant", "content": "Il monitoraggio ambientale con sensori IoT consente di raccogliere dati in tempo reale su temperatura, umidità, qualità dell'aria e altri parametri."},
    {"role": "user", "content": "Quali sono i vantaggi rispetto ai metodi tradizionali?"},
    {"role": "assistant", "content": "I vantaggi principali sono: costi ridotti, copertura capillare del territorio, dati in tempo reale e possibilità di automazione degli interventi."},
]

conversazione_en = [
    {"role": "user", "content": "What is environmental monitoring with IoT sensors?"},
    {"role": "assistant", "content": "Environmental monitoring with IoT sensors enables real-time data collection on temperature, humidity, air quality and other parameters."},
    {"role": "user", "content": "What are the advantages over traditional methods?"},
    {"role": "assistant", "content": "The main advantages are: reduced costs, comprehensive territory coverage, real-time data and the ability to automate interventions."},
]

# TODO: constate i token per entrambe le conversazioni
token_it = ___
token_en = ___

print(f"Token italiano:  {token_it}")
print(f"Token inglese:   {token_en}")
print(f"Differenza:      +{token_it - token_en} token")
print(f"Overhead:        +{((token_it/token_en)-1)*100:.1f}%")
print()
print("💡 Per un chatbot WiData con 1000 conversazioni al mese,")
print("   quanto costa di più usare l'italiano rispetto all'inglese?")
# TODO: calcolate il costo extra mensile
# ...

---
### Esercizio 3 — System prompt lungo vs corto: impatto sulla KV Cache *(libero)*

La KV Cache di Anthropic serve il system prompt identico
al 10% del costo dalla seconda chiamata in poi.

Costruite un sistema prompt lungo (~200 token) e uno corto (~30 token).
Calcolate il risparmio della KV Cache su 100 chiamate per entrambi.

In [ ]:
# Esercizio 3 — KV Cache: quanto si risparmia?

system_lungo = """
# Scrivi qui un system prompt lungo per WiData (almeno 150 parole)
# Includi: ruolo, prodotti, vincoli, tono, formato output, FAQ comuni...
"""

system_corto = """
# Scrivi qui la versione corta (max 30 parole)
"""

N_CHIAMATE = 100

# TODO: constate i token di ogni system prompt
# Hint: usate count_tokens con messages=[{role:user, content:"test"}]
# e system=system_lungo/corto

# TODO: calcolate per ogni system prompt:
# - Costo senza KV Cache (N_CHIAMATE × token_system × $1/M)
# - Costo con KV Cache (prima chiamata normale + resto al 10%)
# - Risparmio totale

# Conclusione:
# Vale la pena ottimizzare il system prompt per la KV Cache?
# In quali scenari il risparmio è significativo?
# ...

---
### Esercizio 4 — Simulare il limite della context window *(libero)*

Costruite una conversazione artificialmente lunga
e osservate cosa succede quando si avvicina al limite.

Quanto si può fare prima che diventi problematico in termini di costo?
Quando conviene iniziare a troncare?

In [ ]:
# Esercizio 4 — simulare una conversazione lunga

# Costruite una conversazione con messaggi lunghi (almeno 100 parole ciascuno)
# Monitorare i token ad ogni turno
# Definire una soglia oltre cui conviene troncare

SOGLIA_TOKEN = 5000  # ← cambiate questo valore
history_lunga = []

# TODO: fate girare la conversazione finché non supera la soglia
# Ad ogni turno stampate: turno, token totali, % della context window usata
# Context window Haiku = 200.000 token

CONTEXT_WINDOW = 200_000

# ...

# Conclusione del gruppo:
# A quanti turni conviene iniziare a troncare per il chatbot WiData?
# ...

---
## 📊 Preparate la presentazione (5 slide)

1. **Cos'è la context window** — con l'analogia del tavolo
2. **Come crescono i token** — con il grafico dei vostri risultati
3. **Italiano vs inglese** — i numeri che avete misurato
4. **KV Cache** — quanto si risparmia con system prompt lungo vs corto
5. **La vostra raccomandazione** — quando troncare per il chatbot WiData

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*